# TrustLens — Production Model Development

## Phase 5D: Train-Serving Consistent Random Forest

This notebook develops the production-ready TrustLens phishing classification model.

Unlike the Phase 4 experimental model, this model is trained only on features generated by the TrustLens production feature extractor.

### Production Pipeline

Raw URL → Production Feature Extractor → 17 Features → Random Forest → Prediction

The objective is to ensure that the same feature definitions are used during both training and real-world inference, preventing train-serving skew.

The Phase 4/4.5 model remains unchanged and serves as the experimental baseline.

In [1]:
import pandas as pd
import numpy as np

production_df = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Dataset shape:", production_df.shape)
print("Total missing values:", production_df.isnull().sum().sum())

print("\nColumns:")
print(production_df.columns.tolist())

print("\nLabel distribution:")
print(production_df["label"].value_counts())

production_df.head()

Dataset shape: (235370, 19)
Total missing values: 0

Columns:
['URL', 'URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'IsHTTPS', 'NoOfDots', 'NoOfSlashes', 'SuspiciousKeywordCount', 'HasHyphenInDomain', 'PathDepth', 'label']

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64


,URL,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth,label
0,https://www.southbankmosaics.com,32,24,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0,1
1,https://www.uni-mainz.de,24,16,0,de,2,1,0,0,0,0,0,1,2,2,0,1,0,1
2,https://www.voicefmradio.co.uk,30,22,0,uk,2,2,0,0,0,0,0,1,3,2,0,0,0,1
3,https://www.sfnmjournal.com,27,19,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0,1
4,https://www.rewildingargentina.org,34,26,0,org,3,1,0,0,0,0,0,1,2,2,0,0,0,1


In [2]:
# Phase 5D - Step 3:
# Define production features (X) and target (y)

PRODUCTION_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

X = production_df[PRODUCTION_FEATURES].copy()
y = production_df["label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature count:", len(PRODUCTION_FEATURES))

print("\nFeature data types:")
print(X.dtypes)

print("\nMissing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

X shape: (235370, 17)
y shape: (235370,)

Feature count: 17

Feature data types:
URLLength                 int64
DomainLength              int64
IsDomainIP                int64
TLD                         str
TLDLength                 int64
NoOfSubDomain             int64
HasObfuscation            int64
NoOfObfuscatedChar        int64
NoOfDegitsInURL           int64
NoOfEqualsInURL           int64
NoOfQMarkInURL            int64
IsHTTPS                   int64
NoOfDots                  int64
NoOfSlashes               int64
SuspiciousKeywordCount    int64
HasHyphenInDomain         int64
PathDepth                 int64
dtype: object

Missing values in X: 0
Missing values in y: 0


In [3]:
# Phase 5D - Step 4:
# Create train/test split BEFORE fitting TLD preprocessing

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

print("\nTrain percentage:", round(len(X_train) / len(X) * 100, 2))
print("Test percentage:", round(len(X_test) / len(X) * 100, 2))

X_train shape: (188296, 17)
X_test shape: (47074, 17)

y_train distribution:
label
1    107880
0     80416
Name: count, dtype: int64

y_test distribution:
label
1    26970
0    20104
Name: count, dtype: int64

Train percentage: 80.0
Test percentage: 20.0


In [4]:
from sklearn.preprocessing import TargetEncoder

print("TargetEncoder imported successfully.")

TargetEncoder imported successfully.


In [5]:
# Phase 5D - Step 5:
# Define production preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

categorical_features = ["TLD"]

numeric_features = [
    feature
    for feature in PRODUCTION_FEATURES
    if feature != "TLD"
]

tld_encoder = TargetEncoder(
    target_type="binary",
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("tld_target_encoder", tld_encoder, categorical_features),
        ("numeric", "passthrough", numeric_features)
    ],
    remainder="drop"
)

print("Categorical features:", categorical_features)
print("Numeric feature count:", len(numeric_features))
print("\nPreprocessor created successfully.")

Categorical features: ['TLD']
Numeric feature count: 16

Preprocessor created successfully.


In [6]:
# Phase 5D - Step 6:
# Build complete production ML pipeline

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

production_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

print("Production pipeline created successfully.")

Production pipeline created successfully.


In [7]:
# Phase 5D - Step 7:
# Train production pipeline using training data only

production_pipeline.fit(X_train, y_train)

print("Production model training completed successfully.")

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Production model training completed successfully.


In [8]:
# Phase 5D - Step 8:
# Evaluate production pipeline on untouched test data

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred = production_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.997
Precision: 0.9957
Recall: 0.999
F1 Score: 0.9974

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      1.00     20104
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion Matrix:
[[19988   116]
 [   26 26944]]


## Production Model Evaluation

The 17-feature train-serving-consistent Random Forest achieved an accuracy of approximately **99.70%** on the untouched test set.

The confusion matrix showed:

- 19,988 phishing URLs correctly classified as phishing.
- 116 phishing URLs incorrectly classified as legitimate.
- 26 legitimate URLs incorrectly classified as phishing.
- 26,944 legitimate URLs correctly classified as legitimate.

These results demonstrate that removing unreproducible features and rebuilding the training dataset using the production feature extractor preserved very strong classification performance while eliminating known train-serving feature inconsistencies.

In [9]:
# Phase 5D - Step 9:
# Calculate phishing-specific metrics (phishing = label 0)

production_phishing_precision = precision_score(
    y_test,
    y_pred,
    pos_label=0
)

production_phishing_recall = recall_score(
    y_test,
    y_pred,
    pos_label=0
)

production_phishing_f1 = f1_score(
    y_test,
    y_pred,
    pos_label=0
)

print("Production Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Phishing Precision:", round(production_phishing_precision, 4))
print("Phishing Recall:", round(production_phishing_recall, 4))
print("Phishing F1-Score:", round(production_phishing_f1, 4))

Production Accuracy: 0.997
Phishing Precision: 0.9987
Phishing Recall: 0.9942
Phishing F1-Score: 0.9965


## Experimental vs Production Model Comparison

| Metric | Phase 4/4.5 Experimental RF | Phase 5 Production RF |
|---|---:|---:|
| Accuracy | 99.79% | 99.70% |
| Phishing Precision | 99.94% | 99.87% |
| Phishing Recall | 99.57% | 99.42% |
| Phishing F1-Score | 99.76% | 99.65% |

The production model shows only a very small reduction in test-set performance compared with the experimental Random Forest.

This reduction is accepted because the production model eliminates known train-serving inconsistencies discovered during the feature reproducibility audit.

The experimental model used 24 URL-related features, while the production model uses 17 features generated consistently by the TrustLens production feature extractor.

Therefore, the 17-feature model is selected as the production model because it provides:

- Strong phishing classification performance
- Reproducible feature definitions
- Consistent training and inference behavior
- Reduced risk of train-serving skew
- A deployable preprocessing and prediction pipeline

The Phase 4/4.5 model is retained as the experimental baseline and is not modified.

In [10]:
# Phase 5D - Step 10:
# Save the complete fitted production pipeline

import joblib
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "trustlens_production_pipeline.joblib"

joblib.dump(
    production_pipeline,
    model_path
)

print("Production pipeline saved successfully.")
print("Saved to:", model_path)

Production pipeline saved successfully.
Saved to: ..\models\trustlens_production_pipeline.joblib


# Phase 5F - Step 1:
# Reload saved production pipeline

loaded_pipeline = joblib.load(
    "../models/trustlens_production_pipeline.joblib"
)

print("Production pipeline reloaded successfully.")

In [12]:
import joblib

loaded_pipeline = joblib.load(
    "../models/trustlens_production_pipeline.joblib"
)

print("Production pipeline reloaded successfully.")

Production pipeline reloaded successfully.


In [13]:
test_url = "https://example-login.com/account/verify?id=123"

features = extract_url_features(test_url)

input_df = pd.DataFrame([features])

prediction = loaded_pipeline.predict(input_df)[0]
probabilities = loaded_pipeline.predict_proba(input_df)[0]

print("URL:", test_url)
print("Prediction:", prediction)
print("Probabilities:", probabilities)

URL: https://example-login.com/account/verify?id=123
Prediction: 0
Probabilities: [1. 0.]


In [14]:
print("Model class order:", loaded_pipeline.classes_)

Model class order: [0 1]


In [15]:
# Phase 5F - Probability distribution analysis

test_probabilities = production_pipeline.predict_proba(X_test)

# Class order is [0, 1]
phishing_probabilities = test_probabilities[:, 0]

print("Minimum phishing probability:", phishing_probabilities.min())
print("Maximum phishing probability:", phishing_probabilities.max())
print("Mean phishing probability:", phishing_probabilities.mean())

print("\nPercentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(
        f"{p}th percentile:",
        round(np.percentile(phishing_probabilities, p), 4)
    )

print("\nProbability band counts:")
print("< 0.40:", (phishing_probabilities < 0.40).sum())
print("0.40 to < 0.75:",
      ((phishing_probabilities >= 0.40) &
       (phishing_probabilities < 0.75)).sum())
print(">= 0.75:", (phishing_probabilities >= 0.75).sum())

Minimum phishing probability: 0.0
Maximum phishing probability: 1.0
Mean phishing probability: 0.42709097167291193

Percentiles:
1th percentile: 0.0
5th percentile: 0.0
10th percentile: 0.0
25th percentile: 0.0
50th percentile: 0.0067
75th percentile: 1.0
90th percentile: 1.0
95th percentile: 1.0
99th percentile: 1.0

Probability band counts:
< 0.40: 27038
0.40 to < 0.75: 67
>= 0.75: 19969


In [17]:
# Phase 5F - Inspect current suspicious / borderline band

borderline_mask = (
    (phishing_probabilities >= 0.40)
    & (phishing_probabilities < 0.75)
)

borderline_results = X_test.loc[borderline_mask].copy()

borderline_results["ActualLabel"] = y_test.loc[
    borderline_mask
].values

borderline_results["PhishingProbability"] = (
    phishing_probabilities[borderline_mask]
)

borderline_results["URL"] = production_df.loc[
    borderline_results.index,
    "URL"
]

borderline_results = borderline_results[
    [
        "URL",
        "ActualLabel",
        "PhishingProbability",
        "TLD",
        "URLLength",
        "SuspiciousKeywordCount",
        "HasHyphenInDomain",
        "PathDepth",
    ]
].sort_values(
    "PhishingProbability"
)

print("Borderline cases:", len(borderline_results))

borderline_results


Borderline cases: 67


,URL,ActualLabel,PhishingProbability,TLD,URLLength,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
73347,https://www.las2orillas.co,1,0.403333,co,26,0,0,0
23200,https://www.hinews.cn,1,0.403341,cn,21,0,0,0
117739,https://www.wooozy.cn,1,0.403341,cn,21,0,0,0
109793,https://www.appleid-find.cloud,0,0.406667,cloud,30,0,1,0
162846,https://www.seofreelancerbangalore.in,1,0.411814,in,37,0,0,0
...,...,...,...,...,...,...,...,...
205634,https://www.icy.lud.workers.dev,0,0.726667,dev,31,0,0,0
234289,https://cr000.bnmovil.repl.co,0,0.730000,co,29,0,0,0
94268,https://iunbtissloit.chantinll.repl.co,0,0.736484,co,38,0,0,0
143963,https://www.reporterosasociados.com.co,1,0.736484,co,38,0,0,0


In [18]:
print("Borderline label distribution:")
print(borderline_results["ActualLabel"].value_counts())

print("\nPercentages:")
print(
    borderline_results["ActualLabel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Borderline label distribution:
ActualLabel
1    39
0    28
Name: count, dtype: int64

Percentages:
ActualLabel
1    58.21
0    41.79
Name: proportion, dtype: float64


## Risk Interpretation Threshold Validation

The production Random Forest remains a binary classifier:

- 0 = Phishing
- 1 = Legitimate

TrustLens adds a separate user-facing risk interpretation layer based on the
predicted phishing probability.

### Risk Bands

- Phishing probability < 0.40 → Low Risk / Legitimate
- Phishing probability 0.40 to < 0.75 → Medium Risk / Suspicious
- Phishing probability >= 0.75 → High Risk / Phishing

The Suspicious category is not a third machine-learning class. It represents
an uncertainty region between confident legitimate and confident phishing
predictions.

On the 47,074-row test set, 67 predictions fell within the 0.40–0.75
uncertainty band.

Among these borderline predictions:

- 39 (58.21%) were actually legitimate.
- 28 (41.79%) were actually phishing.

The presence of both classes within this region supports its interpretation
as an uncertain or suspicious prediction zone.

Only approximately 0.14% of test predictions fell within this band,
indicating that the Random Forest is highly confident for most samples.

These thresholds are used as an interpretable Version 1 decision policy and
can be recalibrated in future versions using additional validation data and
probability-calibration techniques.

In [19]:
# Derive structural thresholds from training data only

structural_features = [
    "URLLength",
    "PathDepth",
    "NoOfSubDomain",
    "NoOfDots"
]

for feature in structural_features:
    print(f"\n=== {feature} ===")
    print("75th percentile:", round(X_train[feature].quantile(0.75), 2))
    print("90th percentile:", round(X_train[feature].quantile(0.90), 2))
    print("95th percentile:", round(X_train[feature].quantile(0.95), 2))
    print("99th percentile:", round(X_train[feature].quantile(0.99), 2))


=== URLLength ===
75th percentile: 35.0
90th percentile: 50.0
95th percentile: 74.0
99th percentile: 147.0

=== PathDepth ===
75th percentile: 0.0
90th percentile: 1.0
95th percentile: 2.0
99th percentile: 4.0

=== NoOfSubDomain ===
75th percentile: 1.0
90th percentile: 2.0
95th percentile: 2.0
99th percentile: 3.0

=== NoOfDots ===
75th percentile: 2.0
90th percentile: 3.0
95th percentile: 3.0
99th percentile: 5.0


## Rule-Based Explanation Threshold Validation

TrustLens provides human-readable URL risk indicators alongside the machine-learning prediction.

Structural explanation thresholds were derived from the training data rather than selected arbitrarily.

### Training-Set Percentile Analysis

| Feature | 95th Percentile | 99th Percentile | Explanation Threshold |
|---|---:|---:|---:|
| URLLength | 74 | 147 | > 75 |
| PathDepth | 2 | 4 | >= 4 |
| NoOfSubDomain | 2 | 3 | >= 3 |
| NoOfDots | 3 | 5 | >= 5 |

The URL-length threshold is positioned just above the 95th percentile, while the path-depth, subdomain, and dot-count thresholds correspond approximately to the 99th percentile of the training data.

These rules are used only to generate interpretable warning indicators for the user. They do not replace the Random Forest classifier and should not be interpreted as the model's internal explanation.